In [ ]:
import json
import pandas as pd
import numpy as np
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import os

warnings.filterwarnings('ignore')

In [ ]:
def process_rehab_json(file_paths):
    dataset_rows = []
    
    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as f:
            docs = json.load(f) 
            
            if isinstance(docs, dict):
                docs = [docs]
                
            for doc in docs:
                planning_date_str = doc.get('planningDate', {}).get('$date')
                if not planning_date_str:
                    continue
                planning_date = pd.to_datetime(planning_date_str).tz_localize(None)
                
                agenda = doc.get('agenda', [])
                if not agenda:
                    continue
                
                # Calcolo del Density Ratio giornaliero
                unique_patients = set()
                unique_operators = set()
                
                # Contiamo quanti pazienti e operatori distinti ci sono in questa giornata/team
                for item in agenda:
                    if 'patient' in item and 'id' in item['patient']:
                        unique_patients.add(item['patient']['id'])
                    if 'operator' in item and 'id' in item['operator']:
                        unique_operators.add(item['operator']['id'])
                        
                num_patients = len(unique_patients)
                num_operators = len(unique_operators)
                
                # Calcolo del rapporto 
                density_ratio = (num_patients / num_operators) if num_operators > 0 else np.nan
                
                for item in agenda:
                    operator = item.get('operator', {})
                    patient = item.get('patient', {})
                    session = item.get('session', {})
                    
                    start_time = pd.to_datetime(item.get('start'), format='%H:%M', errors='coerce')
                    end_time = pd.to_datetime(item.get('end'), format='%H:%M', errors='coerce')
                    duration_target = (end_time - start_time).seconds / 60 if pd.notnull(start_time) and pd.notnull(end_time) else np.nan
                    
                    birth_date_str = patient.get('birthDate')
                    age = np.nan
                    if birth_date_str:
                        birth_date = pd.to_datetime(birth_date_str).tz_localize(None)
                        age = (planning_date - birth_date).days / 365.25
                        
                    row = {
                        'pat_type': patient.get('type'),
                        'pat_autonomous': patient.get('autonomous'),
                        'pat_aidNeeds': patient.get('aidNeeds'),
                        'pat_overallMinLength': patient.get('overallMinLength'),
                        'pat_age': age, 
                        'pat_has_preferred_ops': len(patient.get('preferredOps', [])) > 0, 
                        
                        'op_jobKind': operator.get('jobKind'),
                        'op_burdenScore': operator.get('burdenScore'),
                        'op_qualifications_count': len(operator.get('qualifications', [])), 
                        
                        'sess_minLength': session.get('minLength'),
                        'sess_kind': session.get('kind'),
                        'sess_flexibility_delta': session.get('idealLength', 0) - session.get('minLength', 0), 
                        
                        'density_ratio': density_ratio,
                        'planning_date': planning_date,
                        
                        'target_assignments': duration_target
                    }
                    dataset_rows.append(row)
                    
    return pd.DataFrame(dataset_rows)


json_files = glob.glob("data/*.json")
df = process_rehab_json(json_files)
print(f"Estratte {len(df)} sedute iniziali.")

In [ ]:
def generate_statistical_report(df):
    report_data = []
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        n_unique = df[col].nunique()
        n_missing = df[col].isnull().sum()
        pct_missing = (n_missing / len(df)) * 100
        
        # Analisi della distribuzione
        if pd.api.types.is_numeric_dtype(df[col]) and n_unique > 2:
            # Per le numeriche
            dist_info = f"Min: {df[col].min():.1f} | Media: {df[col].mean():.1f} | Max: {df[col].max():.1f}"
        else:
            # Per categoriche, booleane o numeriche con pochissimi valori
            top_vals = df[col].value_counts().head(3).to_dict()
            dist_info = "Top val: " + ", ".join([f"{k} ({v})" for k, v in top_vals.items()])
            
        report_data.append({
            'Feature': col,
            'Tipo': dtype,
            'Valori Distinti': n_unique,
            'Valori Mancanti': f"{n_missing} ({pct_missing:.1f}%)",
            'Distribuzione': dist_info
        })
        
    report_df = pd.DataFrame(report_data)
    return report_df

stats_report = generate_statistical_report(df)
display(stats_report)

In [ ]:
def initial_hygiene_cleaning(df):
    print(f"Dimensioni originali: {df.shape}")
    
    # Rimozione dei record in cui il target è nullo
    df_clean = df.dropna(subset=['target_assignments']).copy()

    df_clean['target_assignments'] = df_clean['target_assignments'].astype(float)
    print(f"Dimensioni dopo rimozione target mancanti: {df_clean.shape}")
    
    # Identificazione e rimozione delle feature costanti
    constant_cols = [col for col in df_clean.columns if df_clean[col].nunique() <= 1]
    
    if constant_cols:
        df_clean = df_clean.drop(columns=constant_cols)
        print(f"Rimosse feature costanti: {constant_cols}")
    else:
        print("Nessuna feature costante trovata.")
        
    print(f"Dimensioni finali dataset base: {df_clean.shape}")
    return df_clean

# Esecuzione
df_base = initial_hygiene_cleaning(df)

In [ ]:
def chronological_train_test_split(df, train_ratio=0.8):
    # Ordinamento del dataset per data di pianificazione
    df_sorted = df.sort_values('planning_date').reset_index(drop=True)
    
    # Calcolo dell'indice esatto in cui tagliare
    split_idx = int(len(df_sorted) * train_ratio)
    
    # Divisione in Train e Test
    df_train = df_sorted.iloc[:split_idx].copy()
    df_test = df_sorted.iloc[split_idx:].copy()
    
    print(f"Dati totali: {len(df_sorted)}")
    print(f"Train set: {len(df_train)} record (dal {df_train['planning_date'].min().date()} al {df_train['planning_date'].max().date()})")
    print(f"Test set:  {len(df_test)} record (dal {df_test['planning_date'].min().date()} al {df_test['planning_date'].max().date()})")
    
    # Rimozione della colonna data, non più necessaria
    df_train = df_train.drop(columns=['planning_date'])
    df_test = df_test.drop(columns=['planning_date'])
    
    return df_train, df_test

df_train, df_test = chronological_train_test_split(df_base, train_ratio=0.8)

In [ ]:
def apply_iqr_filter_train(df_train, target_col='target_assignments'):
    Q1 = df_train[target_col].quantile(0.25)
    Q3 = df_train[target_col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Maschera per tenere i valori dentro i limiti
    mask = (df_train[target_col] >= lower_bound) & (df_train[target_col] <= upper_bound)
    
    outliers_removed = (~mask).sum()
    df_train_filtered = df_train[mask].copy()
    
    print(f"Train Set: Rimossi {outliers_removed} record anomali per il target '{target_col}'.")
    print(f"Limiti applicati: {lower_bound:.1f} - {upper_bound:.1f} min")
    
    return df_train_filtered

df_train_iqr = apply_iqr_filter_train(df_train, target_col='target_assignments')

In [ ]:
def plot_correlation_matrix(df, title="Matrice di Correlazione", save_dir="images", filename="corr_matrix.png"):
    os.makedirs(save_dir, exist_ok=True)
    
    # Selezioniamo solo le variabili numeriche
    num_df = df.select_dtypes(include=[np.number])
    
    # Calcoliamo la matrice di correlazione di Pearson
    corr_matrix = num_df.corr()
    
    # Impostiamo la dimensione della figura
    plt.figure(figsize=(12, 10))
    
    # Opzionale: Creiamo una maschera per nascondere la metà superiore (poiché è speculare)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    # Disegniamo la heatmap
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    
    plt.title(title, fontsize=16, pad=20)
    plt.tight_layout()
    
    # Salvataggio
    plt.savefig(f"{save_dir}/{filename}", bbox_inches='tight')
    plt.show()

plot_correlation_matrix(df_train_iqr, title="Matrice di Correlazione di Pearson (Train Set)", filename="corr_matrix_pre.png")

In [ ]:
def apply_correlation_filter(df_train, df_test, threshold=0.90, protected_features=['sess_flexibility_delta']):
    # Calcolo correlazione
    num_df = df_train.select_dtypes(include=[np.number])
    corr_matrix = num_df.corr().abs()
    
    # Identificazione feature da rimuovere, considerando le eccezioni
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    to_drop = []
    for column in upper.columns:
        # Se la colonna è fortemente correlata con qualcosa...
        if any(upper[column] > threshold):
            # ...e non è nella lista delle protette, la scartiamo.
            if column not in protected_features:
                to_drop.append(column)
            else:
                # Se è protetta, cerchiamo con cosa è correlata per scartare l'altra variabile
                correlated_with_protected = upper.index[upper[column] > threshold].tolist()
                to_drop.extend([c for c in correlated_with_protected if c not in protected_features])

    # Rimuozione eventuali duplicati dalla lista to_drop
    to_drop = list(set(to_drop))

    if to_drop:
        print(f"Feature rimosse per correlazione > {threshold}: {to_drop}")
        df_train_clean = df_train.drop(columns=to_drop)
        df_test_clean = df_test.drop(columns=to_drop)
    else:
        print(f"Nessuna feature supera la soglia di correlazione ({threshold}).")
        df_train_clean, df_test_clean = df_train.copy(), df_test.copy()
        
    return df_train_clean, df_test_clean

df_train_corr, df_test_corr = apply_correlation_filter(df_train_iqr, df_test, threshold=0.90)

In [ ]:
def group_rare_categories(df_train, df_test, cat_col='pat_type', threshold=0.05):
    df_train_out = df_train.copy()
    df_test_out = df_test.copy()
    
    # Calcolo delle frequenze sul Train Set
    freq = df_train_out[cat_col].value_counts(normalize=True)
    rare_cats = freq[freq < threshold].index.tolist()
    frequent_cats = freq[freq >= threshold].index.tolist()
    
    if rare_cats:
        print(f"Categorie rare individuate (< {threshold*100}%): {rare_cats}")
        
        # Le categorie rare del Train diventano 'Other'
        df_train_out[cat_col] = df_train_out[cat_col].replace(rare_cats, 'Other')
        
        # Sostituiamo nel Test: se un valore non è tra quelli frequenti del Train, diventa 'Other'
        df_test_out[cat_col] = df_test_out[cat_col].apply(lambda x: x if x in frequent_cats else 'Other')
        
    else:
        print("Nessuna categoria rara trovata sotto la soglia.")
        
    print(f"Distribuzione finale nel Train:\n{df_train_out[cat_col].value_counts(normalize=True) * 100}")
    
    return df_train_out, df_test_out

df_train_grouped, df_test_grouped = group_rare_categories(df_train_corr, df_test_corr)

In [ ]:
def safe_one_hot_encoding(df_train, df_test):
    cat_cols = df_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    
    # Esecuzione dell'encoding con get_dummies
    df_train_encoded = pd.get_dummies(df_train, columns=cat_cols, drop_first=True)
    df_test_encoded = pd.get_dummies(df_test, columns=cat_cols, drop_first=True)
    
    # Allineamento del test set al train set. 
    # - Se nel test manca una colonna del train, viene creata con tutti 0.
    # - Se nel test c'è una colonna nuova non vista nel train, viene ignorata/rimossa.
    df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, join='left', axis=1, fill_value=0)
    
    # Conversione booleani in int
    bool_cols = df_train_encoded.select_dtypes(include='bool').columns
    df_train_encoded[bool_cols] = df_train_encoded[bool_cols].astype(int)
    df_test_encoded[bool_cols] = df_test_encoded[bool_cols].astype(int)
    
    # Normalizzazione nomi colonne
    df_train_encoded.columns = df_train_encoded.columns.astype(str)
    df_test_encoded.columns = df_test_encoded.columns.astype(str)
    
    print(f"Dimensioni finali Train: {df_train_encoded.shape}")
    print(f"Dimensioni finali Test: {df_test_encoded.shape}")
    
    return df_train_encoded, df_test_encoded

df_train_final, df_test_final = safe_one_hot_encoding(df_train_grouped, df_test_grouped)

df_train_final.to_csv("data/train_dataset.csv", index=False)
df_test_final.to_csv("data/test_dataset.csv", index=False)
print("Pipeline conclusa. Dati pronti per il training!")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def plot_features(df, features_to_plot=None, title_suffix="", save_dir="images"):
    os.makedirs(save_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")
    
    # Colonne da plottare: se features_to_plot è None, vengono selezionete tutte le features
    cols = features_to_plot if features_to_plot is not None else df.columns
    
    for col in cols:
        if col not in df.columns:
            continue
            
        plt.figure(figsize=(8, 5))
        col_data = df[col].dropna()
        
        if len(col_data) == 0:
            plt.close()
            continue
            
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        is_discrete = df[col].nunique() < 15 # Soglia per trattarla come categorica nel grafico
        

        plot_title = f"{col} {title_suffix}".strip()
        
        if is_numeric and not is_discrete:
            # Istogramma continuo
            sns.histplot(col_data, kde=True, color="skyblue")
            plt.axvline(col_data.mean(), color='red', linestyle='--', label=f'Media: {col_data.mean():.1f}')
            plt.ylabel('Frequenza')
            plt.legend()
        else:
            # Grafico a barre
            ax = sns.countplot(data=df, x=col, palette="Set2", order=df[col].value_counts().index)
            plt.xticks(rotation=45)
            plt.ylabel('Conteggio')
            
            # Etichette sopra le barre
            for p in ax.patches:
                height = int(p.get_height())
                if height > 0:
                    ax.annotate(f'{height}', 
                                (p.get_x() + p.get_width() / 2., height), 
                                ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')
                    
        plt.title(plot_title, fontsize=14, pad=15)
        plt.xlabel(col)
        plt.tight_layout()
        
        file_suffix = f"_{title_suffix.strip().replace(' ', '_')}" if title_suffix else ""
        plt.savefig(f"{save_dir}/{col}{file_suffix}.png")
        plt.show()


# Prima
plot_features(df_base)

# Dopo
features_modificate = ['target_assignments', 'pat_type'] 
plot_features(df_train_grouped, features_to_plot=features_modificate, title_suffix="post")